In [1]:

# Cell 1 â€” Kaggle-specific Installation
!pip install pip3-autoremove -q
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121 -q
!pip install unsloth -q
!pip install transformers datasets trl peft accelerate bitsandbytes -q
!pip install wandb evaluate rouge_score nltk -q

# NEW FIX: Remove torchao to prevent the int1 import crash
!pip uninstall torchao -y

torch 2.10.0+cu128 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cublas-cu12 12.8.4.1 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cuda-cupti-cu12 12.8.90 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cuda-nvrtc-cu12 12.8.93 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cuda-runtime-cu12 12.8.90 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cudnn-cu12 9.10.2.21 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cufft-cu12 11.3.3.83 (/usr/local/lib/python3.12/dist-packages)
        nvidia-nvjitlink-cu12 12.8.93 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cufile-cu12 1.13.1.3 (/usr/local/lib/python3.12/dist-packages)
    nvidia-curand-cu12 10.3.9.90 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cusolver-cu12 11.7.3.90 (/usr/local/lib/python3.12/dist-packages)
        nvidia-cusparse-cu12 12.5.8.93 (/usr/local/lib/python3.12/dist-packages)
    nvidia-cusparselt-cu12 0.7.1 (/usr/local/lib/python3.12/dist-packages)
    nvidia-nvshmem-cu1

In [2]:
# Cell 2 â€” Kaggle Secrets for Login
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

user_secrets = UserSecretsClient()

# Pull keys securely from Kaggle Add-ons > Secrets
wandb_key = user_secrets.get_secret("WANDB_API_KEY")
hf_key = user_secrets.get_secret("HF_TOKEN")

wandb.login(key=wandb_key)
login(token=hf_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumarsarthakofficial (kumarsarthakofficial-birla-institute-of-technology-mesra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
!pip install --upgrade --no-cache-dir --no-deps unsloth_zoo

In [4]:

# Cell 3 — Load Llama 3.2-3B with stronger LoRA config
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

# KEY UPGRADE: r=64 (was 16) — much more expressive for domain tasks
model = FastLanguageModel.get_peft_model(
    model,
    r=64,                    # Higher rank = more capacity to learn domain knowledge
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,          # alpha = 2x rank is a common best practice
    lora_dropout=0.05,       # Small dropout for regularization
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)



🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [5]:
# Cell 4 - Load high-quality medical reasoning datasets
from datasets import load_dataset, concatenate_datasets
import json

# Load MedQA (USMLE Board Questions) - High quality clinical reasoning
print("Downloading MedQA-USMLE...")
medqa = load_dataset("GBaker/MedQA-USMLE-4-options", split="train")

# Keep PubMedQA - biomedical research Q&A for variety
print("Downloading PubMedQA...")
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

print(f"\nMedQA (USMLE) dataset: {len(medqa)} examples")
print(f"PubMedQA dataset: {len(pubmedqa)} examples")

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

phrases_no_exclude_train.jsonl:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

phrases_no_exclude_test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1273 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]


MedQA (USMLE) dataset: 10178 examples
PubMedQA dataset: 1000 examples


In [6]:
# Cell 5 - Format into ChatML and create a balanced subset

def format_medqa(example):
    """Format USMLE multiple choice questions into ChatML."""
    options_text = ""
    for key, val in example['options'].items():
        options_text += f"{key}. {val}\n"
        
    prompt = f"Question: {example['question']}\n\nOptions:\n{options_text}"
    
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert medical AI assistant. Carefully analyze the clinical presentation and select the most appropriate option."
            },
            {
                "role": "user",
                "content": prompt.strip()
            },
            {
                "role": "assistant",
                "content": f"The correct answer is {example['answer_idx']}: {example['answer']}."
            }
        ]
    }

def format_pubmedqa(example):
    """Format PubMedQA - it has context, question, and long_answer."""
    context = " ".join(example["context"]["contexts"][:2])
    return {
        "messages": [
            {
                "role": "system",
                "content": "You are a knowledgeable medical AI assistant specializing in biomedical research. Provide accurate, evidence-based answers."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {example['question']}"
            },
            {
                "role": "assistant",
                "content": example["long_answer"]
            }
        ]
    }

# Process datasets
medqa_formatted = medqa.map(format_medqa, remove_columns=medqa.column_names)
pubmedqa_formatted = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa.column_names)

# Take 8,000 USMLE questions and the PubMedQA questions to fit within Kaggle T4 limits
medqa_subset = medqa_formatted.shuffle(seed=42).select(range(8000))
sft_dataset = concatenate_datasets([medqa_subset, pubmedqa_formatted]).shuffle(seed=42)

print(f"\nFinal SFT dataset size: {len(sft_dataset)} examples")

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Final SFT dataset size: 9000 examples


In [7]:
# Cell 6 â€” Apply chat template (this is how production models format data)
def apply_chat_template(example):
    """Apply the model's built-in chat template to messages"""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

sft_dataset = sft_dataset.map(apply_chat_template)

# Train/val split â€” CRITICAL for evaluating overfitting
split = sft_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data = split["test"]

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
print("\nFormatted training example:")
print(train_data[0]["text"][:500])

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Train: 8100 | Val: 900

Formatted training example:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Mar 2026

You are a knowledgeable medical AI assistant specializing in biomedical research. Provide accurate, evidence-based answers.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context: It is widely accepted that exemplary surgical care involves a surgeon's involvement in the preoperative, perioperative, and postoperative periods. In an era of ever-expanding therapeutic modal


In [8]:
# Cell 7 - Configure and run SFT with T4-safe settings
import gc
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Initialize WandB run - this is your experiment tracker
wandb.init(
    project="medical-llm-finetune",
    name="llama-3.2-3b-medical-sft-t4-safe",
    config={
        "model": MODEL_NAME,
        "dataset": "medical-qa-shared-task-v1-toy+PubMedQA",
        "lora_r": 32,
        "lora_alpha": 64,
        "max_seq_length": MAX_SEQ_LEN,
        "training_samples": len(train_data),
    }
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,   
        warmup_steps=5,
        average_tokens_across_devices=False,
        num_train_epochs=3,
        learning_rate=2e-4,
        neftune_noise_alpha=5.0,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_torch",             # FIX: Use native PyTorch optimizer to bypass the scaling bug
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        train_sampling_strategy="group_by_length",
        seed=42,
        output_dir="./sft_output",
        report_to="wandb",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        max_grad_norm=0.0,
    ),
)

gc.collect()
torch.cuda.empty_cache()

# --- THE STRONGER BYPASS ---
import transformers.trainer
import transformers.utils.import_utils

# We must patch the function directly inside the trainer's namespace
transformers.trainer.check_torch_load_is_safe = lambda: None
transformers.utils.import_utils.check_torch_load_is_safe = lambda: None
# ---------------------------

print("Starting SFT training...")
sft_stats = sft_trainer.train()
print(f"\nSFT Complete! Final train loss: {sft_stats.training_loss:.4f}")

wandb.finish()

wandb: setting up run zpi325sz
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260327_173423-zpi325sz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run llama-3.2-3b-medical-sft-t4-safe
wandb: ⭐️ View project at https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune
wandb: 🚀 View run at https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune/runs/zpi325sz
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/8100 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/900 [00:00<?, ? examples/s]

Starting SFT training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,100 | Num Epochs = 3 | Total steps = 1,521
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss,Validation Loss
50,1.074513,2.449799
100,1.046056,2.396141
150,1.033124,2.314418
200,1.039848,2.295805
250,1.045374,2.289351
300,1.015821,2.267149
350,1.073199,2.252360
400,1.020778,2.247925
450,1.018109,2.219538
500,1.019795,2.207511


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


SFT Complete! Final train loss: 0.9250


wandb: uploading history steps 181-182, summary, console lines 173-174
wandb: 
wandb: Run history:
wandb:               eval/loss █▇▅▄▄▄▃▃▂▂▃▃▂▂▂▂▂▁▁▁▄▄▄▄▄▄▄▄▄▄
wandb:            eval/runtime ▁▆▆▅▆▆▆▆▆▆▇▆▆█▅▇▇▆▄▇▆▆▆▇▇▆▇██▅
wandb: eval/samples_per_second █▃▃▄▃▃▃▃▃▃▂▃▃▁▄▂▂▃▄▂▃▃▃▂▂▃▂▁▁▄
wandb:   eval/steps_per_second █▃▃▃▃▃▃▃▃▃▂▃▃▁▄▂▂▃▅▂▃▃▃▂▂▃▂▁▁▄
wandb:             train/epoch ▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇██
wandb:       train/global_step ▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
wandb:         train/grad_norm ▆▅▅▃▃▃▃▂▃▃▃▄▃▂▃▄▁▆▄▅▄▃▄▆▄▅▅▆▅▆█▅▆▇▇▇█▇██
wandb:     train/learning_rate ██████████▇▇▇▆▆▆▅▅▅▅▅▅▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
wandb:              train/loss █▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▃▄▃▃▃▃▂▂▁▂▁▂▂▂▂▂▂▂▁
wandb: 
wandb: Run summary:
wandb:               eval/loss 2.29028
wandb:            eval/runtime 161.2918
wandb: eval/samples_per_second 5.58
wandb:   eval/steps_per_second 2.79
wandb:              total_flos 1.171602925503406e+17
wandb:             train/epoch 3
wandb:       train/global_st

In [9]:
# Cell 8 - Evaluate with ROUGE and perplexity
import math
import warnings
import subprocess
import sys
import torch
import gc
from unsloth import FastLanguageModel

try:
    import evaluate
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "evaluate", "rouge_score"])
    import evaluate

from tqdm import tqdm

warnings.filterwarnings(
    "ignore",
    message="The attention mask API under `transformers.modeling_attn_mask_utils`",
    category=FutureWarning,
)
warnings.filterwarnings(
    "ignore",
    message="Both `max_new_tokens`",
    category=UserWarning,
)

rouge = evaluate.load("rouge")

# Ensure tokenizer has a pad token for the perplexity batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_response(messages, max_new_tokens=150):
    """Generate a deterministic response for evaluation with strict memory management."""
    FastLanguageModel.for_inference(model)
    model.eval()

    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    )

    if isinstance(tokenized, torch.Tensor):
        input_ids = tokenized.to("cuda")
        attention_mask = None
    else:
        tokenized = tokenized.to("cuda")
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=False, # cache off saves VRAM
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = input_ids.shape[-1]
    new_tokens = outputs[0][prompt_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # CRITICAL MEMORY CLEANUP
    del input_ids, outputs, tokenized
    if attention_mask is not None:
        del attention_mask
    torch.cuda.empty_cache()
    
    return response

def compute_perplexity(dataset, batch_size=1):
    """Compute perplexity manually to avoid Trainer callback issues, with VRAM flushing."""
    FastLanguageModel.for_inference(model)
    model.eval()

    losses = []
    for start_idx in tqdm(range(0, len(dataset), batch_size), desc="Perplexity"):
        batch = dataset[start_idx:start_idx + batch_size]["text"]
        encodings = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
        ).to("cuda")

        labels = encodings["input_ids"].clone()
        labels[encodings["attention_mask"] == 0] = -100

        with torch.no_grad():
            outputs = model(**encodings, labels=labels)
            loss_val = outputs.loss.item()
            losses.append(loss_val)

        # CRITICAL MEMORY CLEANUP
        del encodings, labels, outputs
        if start_idx % 10 == 0:  # Flush VRAM every 10 steps to prevent fragmentation
            torch.cuda.empty_cache()
            gc.collect()

    mean_loss = sum(losses) / len(losses)
    return mean_loss, math.exp(mean_loss)

# Manual perplexity from the validation set
val_loss, perplexity = compute_perplexity(val_data)

# Run ROUGE on a small validation sample
sample_size = min(50, len(val_data))
eval_sample = val_data.select(range(sample_size))
predictions = []
references = []

print("Running ROUGE evaluation...")
for example in tqdm(eval_sample, desc="ROUGE"):
    messages = example["messages"]
    ref_answer = messages[2]["content"]

    prompt_msgs = [messages[0], messages[1]]
    pred = generate_response(prompt_msgs, max_new_tokens=150)
    predictions.append(pred)
    references.append(ref_answer)

results = rouge.compute(predictions=predictions, references=references)
print("\n=== EVALUATION RESULTS ===")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Perplexity: {perplexity:.4f}")
print(f"ROUGE-1: {results['rouge1']:.4f}")
print(f"ROUGE-2: {results['rouge2']:.4f}")
print(f"ROUGE-L: {results['rougeL']:.4f}")
print(f"\nSample prediction:\n{predictions[0][:300]}")
print(f"\nSample reference:\n{references[0][:300]}")

Perplexity: 100%|██████████| 900/900 [03:53<00:00,  3.86it/s]


Running ROUGE evaluation...


ROUGE: 100%|██████████| 50/50 [02:39<00:00,  3.19s/it]


=== EVALUATION RESULTS ===
Validation Loss: 1.0827
Perplexity: 2.9527
ROUGE-1: 0.7866
ROUGE-2: 0.7148
ROUGE-L: 0.7705

Sample prediction:
The study shows that post-tonsillectomy late haemorrhage is more frequently observed in the night-time. The authors suggest that the night-time haemorrhage may be a sign of a more serious condition.

Sample reference:
The incidence of post-tonsillectomy late haemorrhage in our study population was 1.78%. A statistically significant difference was found between night-time and day-time haemorrhages. Even though no significant distribution of haemorrhages per hour was observed, we underline that we recorded 32 (54.2


In [10]:
# Cell 9 - Build a stronger DPO preference dataset
import random
import re


dpo_source = train_data.select(range(min(500, len(train_data))))
answer_pool = [
    example["messages"][2]["content"].strip()
    for example in dpo_source
    if example["messages"][2]["content"].strip()
]


def sample_mismatched_answer(chosen_response):
    candidates = [answer for answer in answer_pool if answer != chosen_response]
    return random.choice(candidates) if candidates else "Please consult a healthcare professional for personalized advice."


def make_incomplete_answer(chosen_response):
    sentences = [sentence.strip() for sentence in re.split(r"(?<=[.!?])\s+", chosen_response.strip()) if sentence.strip()]
    if len(sentences) >= 2:
        return sentences[0] + " Please discuss the remaining details with a clinician."

    words = chosen_response.split()
    shortened = " ".join(words[: min(20, len(words))]).strip()
    if shortened and shortened[-1] not in ".!?":
        shortened += "."
    return shortened + " Please discuss the remaining details with a clinician."


def make_overcautious_answer(chosen_response):
    return "This depends heavily on the patient and clinical context. A clinician should review the case directly before any conclusion is made."


def make_binary_flip_answer(chosen_response):
    stripped = chosen_response.strip()
    lowered = stripped.lower()
    if lowered.startswith("yes"):
        return re.sub(r"^[Yy]es", "No", stripped, count=1)
    if lowered.startswith("no"):
        return re.sub(r"^[Nn]o", "Yes", stripped, count=1)
    return None


def create_dpo_pair(example):
    """Create harder prompt / chosen / rejected triples for DPO training."""
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(
        [messages[0], messages[1]],
        tokenize=False,
        add_generation_prompt=True,
    )
    chosen_response = messages[2]["content"].strip()

    strategies = [
        ("mismatched_answer", lambda: sample_mismatched_answer(chosen_response)),
        ("incomplete_answer", lambda: make_incomplete_answer(chosen_response)),
        ("overcautious_answer", lambda: make_overcautious_answer(chosen_response)),
    ]

    flipped_answer = make_binary_flip_answer(chosen_response)
    if flipped_answer is not None and flipped_answer != chosen_response:
        strategies.append(("binary_flip", lambda: flipped_answer))

    strategy_name, strategy_fn = random.choice(strategies)
    rejected_response = strategy_fn().strip()
    if rejected_response == chosen_response:
        strategy_name = "mismatched_answer"
        rejected_response = sample_mismatched_answer(chosen_response)

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
        "rejection_strategy": strategy_name,
    }


dpo_dataset = dpo_source.map(
    create_dpo_pair,
    remove_columns=dpo_source.column_names,
)

print(f"DPO dataset: {len(dpo_dataset)} preference pairs")
print("\nRejection strategy counts:")
strategy_counts = {}
for row in dpo_dataset:
    strategy_counts[row["rejection_strategy"]] = strategy_counts.get(row["rejection_strategy"], 0) + 1
print(strategy_counts)

print("\nSample DPO pair:")
print(f"Prompt: {dpo_dataset[0]['prompt'][:120]}...")
print(f"Chosen: {dpo_dataset[0]['chosen'][:120]}...")
print(f"Rejected ({dpo_dataset[0]['rejection_strategy']}): {dpo_dataset[0]['rejected'][:160]}...")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DPO dataset: 500 preference pairs

Rejection strategy counts:
{'overcautious_answer': 164, 'mismatched_answer': 178, 'incomplete_answer': 158}

Sample DPO pair:
Prompt: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Mar 20...
Chosen: The continuity-of-care experiences of vascular trainees are suboptimal. This is especially true for postoperative clinic...
Rejected (overcautious_answer): This depends heavily on the patient and clinical context. A clinician should review the case directly before any conclusion is made....


In [11]:
# Cell 10 - Run DPO training without TRL dependency issues
import gc
import random
import torch
import torch.nn.functional as F
from tqdm import tqdm

BETA = 0.2
DPO_LEARNING_RATE = 5e-6
DPO_GRAD_ACCUM = 8
DPO_EPOCHS = 1
REFERENCE_ADAPTER_NAME = "reference"

if wandb.run is not None:
    wandb.finish()

model.save_pretrained("./sft_adapter_checkpoint")
tokenizer.save_pretrained("./sft_adapter_checkpoint")
tokenizer.pad_token = tokenizer.eos_token

if not hasattr(model, "load_adapter") or not hasattr(model, "set_adapter"):
    raise RuntimeError("This DPO fallback expects a PEFT model with load_adapter() and set_adapter().")


def get_active_adapter_name(peft_model):
    active_adapter = getattr(peft_model, "active_adapter", None)
    if isinstance(active_adapter, str) and active_adapter:
        return active_adapter

    active_adapters = getattr(peft_model, "active_adapters", None)
    if isinstance(active_adapters, str) and active_adapters:
        return active_adapters
    if isinstance(active_adapters, (list, tuple)) and len(active_adapters) > 0:
        return active_adapters[0]

    return "default"


def set_adapter_safe(adapter_name, inference_mode=False):
    try:
        model.set_adapter(adapter_name, inference_mode=inference_mode)
    except TypeError:
        model.set_adapter(adapter_name)
        if hasattr(model, "set_requires_grad"):
            model.set_requires_grad(adapter_name, not inference_mode)


TRAIN_ADAPTER_NAME = get_active_adapter_name(model)
EXISTING_ADAPTERS = list(getattr(model, "peft_config", {}).keys()) if isinstance(getattr(model, "peft_config", None), dict) else []
if REFERENCE_ADAPTER_NAME not in EXISTING_ADAPTERS:
    model.load_adapter(
        "./sft_adapter_checkpoint",
        adapter_name=REFERENCE_ADAPTER_NAME,
        is_trainable=False,
    )


def build_inputs(prompt_text, response_text):
    full_text = prompt_text + response_text
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )["input_ids"]
    encodings = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to("cuda")

    labels = encodings["input_ids"].clone()
    prompt_length = min(len(prompt_ids), labels.shape[1] - 1)
    labels[:, :prompt_length] = -100
    return encodings, labels


def sequence_logprob_for_active_adapter(prompt_text, response_text):
    encodings, labels = build_inputs(prompt_text, response_text)
    outputs = model(**encodings)

    logits = outputs.logits[:, :-1, :]
    shifted_labels = labels[:, 1:]
    loss_mask = shifted_labels != -100

    safe_labels = shifted_labels.clone()
    safe_labels[~loss_mask] = 0

    token_logps = F.log_softmax(logits, dim=-1).gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
    sequence_logp = (token_logps * loss_mask).sum(dim=-1)
    return sequence_logp


# Convert to a mutable list so we can cache reference log-probs
preference_examples = [dpo_dataset[i] for i in range(len(dpo_dataset))]

print("Precomputing reference log-probs...")
set_adapter_safe(REFERENCE_ADAPTER_NAME, inference_mode=True)
model.eval()
for example in tqdm(preference_examples, desc="Reference"):
    with torch.no_grad():
        example["ref_chosen_logp"] = sequence_logprob_for_active_adapter(example["prompt"], example["chosen"]).item()
        example["ref_rejected_logp"] = sequence_logprob_for_active_adapter(example["prompt"], example["rejected"]).item()

gc.collect()
torch.cuda.empty_cache()

set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)
if hasattr(model, "for_training"):
    try:
        model.for_training(use_gradient_checkpointing=True)
    except TypeError:
        model.for_training()
model.train()
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

trainable_params = [p for p in model.parameters() if p.requires_grad]
try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(trainable_params, lr=DPO_LEARNING_RATE)
    print("Using bitsandbytes AdamW8bit optimizer")
except Exception:
    optimizer = torch.optim.AdamW(trainable_params, lr=DPO_LEARNING_RATE)
    print("Using torch.optim.AdamW optimizer")

wandb.init(
    project="medical-llm-finetune",
    name="llama-3.2-3b-medical-dpo-custom",
    config={
        "phase": "DPO-custom",
        "beta": BETA,
        "dpo_pairs": len(preference_examples),
        "max_seq_length": MAX_SEQ_LEN,
        "learning_rate": DPO_LEARNING_RATE,
        "gradient_accumulation_steps": DPO_GRAD_ACCUM,
    },
)

optimizer.zero_grad()
global_step = 0
running_loss = 0.0

print("Starting custom DPO training...")
for epoch in range(DPO_EPOCHS):
    random.shuffle(preference_examples)
    progress_bar = tqdm(range(len(preference_examples)), desc=f"DPO Epoch {epoch + 1}/{DPO_EPOCHS}")

    for idx in progress_bar:
        example = preference_examples[idx]
        set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)

        chosen_logp = sequence_logprob_for_active_adapter(example["prompt"], example["chosen"])
        rejected_logp = sequence_logprob_for_active_adapter(example["prompt"], example["rejected"])

        ref_chosen = torch.tensor(example["ref_chosen_logp"], device=chosen_logp.device)
        ref_rejected = torch.tensor(example["ref_rejected_logp"], device=chosen_logp.device)

        preference_logit = BETA * ((chosen_logp - rejected_logp) - (ref_chosen - ref_rejected))
        loss = -F.logsigmoid(preference_logit).mean()

        (loss / DPO_GRAD_ACCUM).backward()
        running_loss += loss.item()

        should_step = ((idx + 1) % DPO_GRAD_ACCUM == 0) or (idx == len(preference_examples) - 1)
        if should_step:
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

            average_loss = running_loss / (idx + 1)
            progress_bar.set_postfix(loss=f"{average_loss:.4f}")
            if global_step % 10 == 0:
                wandb.log({
                    "dpo/loss": loss.item(),
                    "dpo/avg_loss": average_loss,
                    "dpo/global_step": global_step,
                    "epoch": epoch + 1,
                })

gc.collect()
torch.cuda.empty_cache()
set_adapter_safe(TRAIN_ADAPTER_NAME, inference_mode=False)
model.save_pretrained("./dpo_output")
tokenizer.save_pretrained("./dpo_output")

final_loss = running_loss / max(len(preference_examples), 1)
print(f"Custom DPO training complete! Final average loss: {final_loss:.4f}")
wandb.finish()

Precomputing reference log-probs...


Reference: 100%|██████████| 500/500 [03:37<00:00,  2.30it/s]


Using bitsandbytes AdamW8bit optimizer


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260327_213245-qwt5po21
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run llama-3.2-3b-medical-dpo-custom
wandb: ⭐️ View project at https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune
wandb: 🚀 View run at https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune/runs/qwt5po21


Starting custom DPO training...


DPO Epoch 1/1:   2%|▏         | 10/500 [00:12<09:48,  1.20s/it, loss=0.6921]

Unsloth: Will smartly offload gradients to save VRAM!


DPO Epoch 1/1: 100%|██████████| 500/500 [10:30<00:00,  1.26s/it, loss=0.1019]
wandb: updating run metadata


Custom DPO training complete! Final average loss: 0.1019


wandb: uploading console lines 3-3
wandb: 
wandb: Run history:
wandb:    dpo/avg_loss █▅▃▂▂▁
wandb: dpo/global_step ▁▂▄▅▇█
wandb:        dpo/loss ▁█▁▁▇▇
wandb:           epoch ▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:    dpo/avg_loss 0.1052
wandb: dpo/global_step 60
wandb:        dpo/loss 0.14423
wandb:           epoch 1
wandb: 
wandb: 🚀 View run llama-3.2-3b-medical-dpo-custom at: https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune/runs/qwt5po21
wandb: ⭐️ View project at: https://wandb.ai/kumarsarthakofficial-birla-institute-of-technology-mesra/medical-llm-finetune
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260327_213245-qwt5po21/logs


In [12]:
# Cell 11 — Push to Hub and Export GGUF
from unsloth import FastLanguageModel

HF_USERNAME = "kumarsarthak98" 
REPO_NAME = f"{HF_USERNAME}/medical-llama-3.2-3b-sft-dpo"

print("1. Pushing merged 16-bit model to Hugging Face...")
model.push_to_hub_merged(REPO_NAME, tokenizer, save_method="merged_16bit", token=True)

print("2. Pushing 4-bit GGUF version (for local Mac/PC deployment)...")
model.push_to_hub_gguf(
    REPO_NAME,
    tokenizer,
    quantization_method="q4_k_m",
    token=True
)
print("✅ All deployments complete!")

1. Pushing merged 16-bit model to Hugging Face...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Saving to kumarsarthak98/medical-llama-3.2-3b-sft-dpo will fail, but using a temp folder works! Switching to a temp folder then uploading!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

ReadTimeout: The read operation timed out